# Notebook 02: Data Preprocessing

## Objective

This notebook prepares the IEEE-CIS Fraud Detection dataset for model development. It handles missing values, removes features with excessive missingness, encodes categorical variables, separates the target variable, and creates training and testing datasets for the TrustLens project.

In [23]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_transaction = pd.read_csv(DATA_DIR / "train_transaction.csv")
train_identity = pd.read_csv(DATA_DIR / "train_identity.csv")

df = train_transaction.merge(train_identity, on="TransactionID", how="left")

print("Merged dataset shape:", df.shape)

Merged dataset shape: (590540, 434)


In [24]:
# Cell 2 → remove high-missing features:

target_col = "isFraud"

missing_percentage = df.isnull().mean() * 100

# Drop features with more than 90% missing values
high_missing_cols = missing_percentage[missing_percentage > 90].index.tolist()

# Do not drop target if accidentally included
high_missing_cols = [col for col in high_missing_cols if col != target_col]

df_reduced = df.drop(columns=high_missing_cols)

print("Number of columns dropped:", len(high_missing_cols))
print("Remaining shape:", df_reduced.shape)
print("Dropped columns:", high_missing_cols)

Number of columns dropped: 12
Remaining shape: (590540, 422)
Dropped columns: ['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']


In [25]:
# Cell 3 — separate features and target:

X = df_reduced.drop(columns=[target_col])
y = df_reduced[target_col]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("Target distribution:")
print(y.value_counts(normalize=True) * 100)

Feature shape: (590540, 421)
Target shape: (590540,)
Target distribution:
isFraud
0    96.500999
1     3.499001
Name: proportion, dtype: float64


In [26]:
# Cell 4 — identify feature types:

numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 392
Categorical columns: 29


/var/folders/0n/bc06zqzx6c3d6nyt08994pfc0000gn/T/ipykernel_67719/89919704.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


In [27]:
# Cell 5 — train/test split:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("y_test distribution:")
print(y_test.value_counts(normalize=True) * 100)

X_train: (472432, 421)
X_test: (118108, 421)
y_train distribution:
isFraud
0    96.501084
1     3.498916
Name: proportion, dtype: float64
y_test distribution:
isFraud
0    96.50066
1     3.49934
Name: proportion, dtype: float64


In [28]:
# Cell 6 — Impute numeric columns using median values learned from the training set

numeric_imputer = SimpleImputer(strategy="median")

X_train_num = pd.DataFrame(
    numeric_imputer.fit_transform(X_train[numeric_cols]),
    columns=numeric_cols,
    index=X_train.index
)

X_test_num = pd.DataFrame(
    numeric_imputer.transform(X_test[numeric_cols]),
    columns=numeric_cols,
    index=X_test.index
)

print("Numeric train shape:", X_train_num.shape)
print("Numeric test shape:", X_test_num.shape)
print("Remaining numeric missing values in train:", X_train_num.isnull().sum().sum())
print("Remaining numeric missing values in test:", X_test_num.isnull().sum().sum())

Numeric train shape: (472432, 392)
Numeric test shape: (118108, 392)
Remaining numeric missing values in train: 0
Remaining numeric missing values in test: 0


In [29]:
# Cell 7 — Impute categorical columns with a placeholder, then ordinal encode

categorical_imputer = SimpleImputer(strategy="constant", fill_value="missing")

X_train_cat_imputed = pd.DataFrame(
    categorical_imputer.fit_transform(X_train[categorical_cols]),
    columns=categorical_cols,
    index=X_train.index
)

X_test_cat_imputed = pd.DataFrame(
    categorical_imputer.transform(X_test[categorical_cols]),
    columns=categorical_cols,
    index=X_test.index
)

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

X_train_cat = pd.DataFrame(
    encoder.fit_transform(X_train_cat_imputed),
    columns=categorical_cols,
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    encoder.transform(X_test_cat_imputed),
    columns=categorical_cols,
    index=X_test.index
)

print("Categorical train shape:", X_train_cat.shape)
print("Categorical test shape:", X_test_cat.shape)
print("Remaining categorical missing values in train:", X_train_cat.isnull().sum().sum())
print("Remaining categorical missing values in test:", X_test_cat.isnull().sum().sum())

Categorical train shape: (472432, 29)
Categorical test shape: (118108, 29)
Remaining categorical missing values in train: 0
Remaining categorical missing values in test: 0


In [30]:
# Cell 8 — Combine processed numeric and categorical features

X_train_processed = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_processed = pd.concat([X_test_num, X_test_cat], axis=1)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

print("Total missing values in processed train:", X_train_processed.isnull().sum().sum())
print("Total missing values in processed test:", X_test_processed.isnull().sum().sum())

Processed train shape: (472432, 421)
Processed test shape: (118108, 421)
Total missing values in processed train: 0
Total missing values in processed test: 0


In [9]:
# Cell 9 — Save processed datasets for model training

X_train_processed.to_csv(PROCESSED_DIR / "X_train_processed.csv", index=False)
X_test_processed.to_csv(PROCESSED_DIR / "X_test_processed.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [33]:
test_transaction_ids = X_test["TransactionID"].copy()

In [ ]:
X_test_processed_with_id = X_test_processed.copy()

X_test_processed_with_id.insert(
    0,
    "TransactionID",
    test_transaction_ids.reset_index(drop=True)
)

In [46]:
X_test_processed_with_id.to_csv(
    ROOT_DIR
    / "data"
    / "processed"
    / "X_test_processed_with_id.csv",
    index=False
)

In [47]:
check_df = pd.read_csv(
    ROOT_DIR
    / "data"
    / "processed"
    / "X_test_processed_with_id.csv",
    nrows=5
)

print("Total columns:", check_df.shape[1])

print(
    "TransactionID included:",
    "TransactionID" in check_df.columns
)

print(
    "Predictive feature count:",
    check_df.drop(
        columns=["TransactionID"]
    ).shape[1]
)

Total columns: 421
TransactionID included: True
Predictive feature count: 420


In [37]:
print("Shape:", X_test_processed_with_id.shape)
print(
    "TransactionID included:",
    "TransactionID" in X_test_processed_with_id.columns
)
print(
    "Predictive features:",
    X_test_processed_with_id.drop(
        columns=["TransactionID"]
    ).shape[1]
)

Shape: (118108, 421)
TransactionID included: True
Predictive features: 420


In [40]:
from pathlib import Path

# Define the root directory safely
ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent



In [48]:
demo_df = pd.read_csv(
    ROOT_DIR
    / "data"
    / "processed"
    / "X_test_processed_with_id.csv"
).sample(
    n=100,
    random_state=42
)

demo_path = (
    ROOT_DIR
    / "data"
    / "processed"
    / "trustlens_demo_sample.csv"
)

demo_df.to_csv(
    demo_path,
    index=False
)

print("Demo shape:", demo_df.shape)

print(
    "TransactionID included:",
    "TransactionID" in demo_df.columns
)

print(
    "Model feature count:",
    demo_df.drop(
        columns=["TransactionID"]
    ).shape[1]
)

Demo shape: (100, 421)
TransactionID included: True
Model feature count: 420
